In [1]:
%load_ext autoreload
%autoreload 2
%cd /home/albin/egna_proj/block_puzzle_rl/

/home/albin/egna_proj/block_puzzle_rl


/home/albin/egna_proj/block_puzzle_rl/.venv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
# ⬇️ SB3 + Gym imports
import numpy as np
import torch
import gymnasium as gym
from stable_baselines3 import DQN
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.monitor import Monitor
from gymnasium import spaces
import numpy as np
from collections import deque
import torch as th
import torch.nn as nn
from stable_baselines3 import PPO
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv

from sb3_contrib import MaskablePPO
from sb3_contrib.common.maskable.utils import get_action_masks
from sb3_contrib.common.wrappers import ActionMasker

# ⬇️ Your imports
from game.block_puzzle_env import BlockPuzzleEnv
from agent.utils import encode_state, encode_state_cnn
from game.block import random_block

In [3]:

# ⬇️ Custom wrapper to convert dict obs → flat array
class FlattenedBlockEnv(gym.Env):
    def __init__(self, width=12, height=12, num_blocks=3):
        super().__init__()
        self.raw_env = BlockPuzzleEnv(width, height, num_blocks)
        self.action_space = self.raw_env.action_space
        dummy_obs, _ = self.raw_env.reset()
        sample_obs = encode_state_cnn(dummy_obs)
        self.observation_space = gym.spaces.Box(
            low=0, high=255, shape=sample_obs.shape, dtype=np.uint8
        )
    
    def reset(self, seed=None, options=None):
        obs_dict, _ = self.raw_env.reset()
        flat_obs = encode_state_cnn(obs_dict)
        return flat_obs.astype(np.uint8), {}

    def step(self, action):
        # Handle batched action from DummyVecEnv (e.g., [0, 9, 10])
        if isinstance(action, (list, np.ndarray)) and len(action) == 3:
            a0, a1, a2 = map(int, action)
        else:
            # Flat index → unravel into (block_index, row, col)
            a0, a1, a2 = np.unravel_index(action, self.action_space.nvec)

        raw_action = (a0, a1, a2)
        (obs_dict, _), reward, terminated, truncated, info = self.raw_env.step(raw_action)
        flat_obs = encode_state_cnn(obs_dict)
        return flat_obs.astype(np.float32), reward, terminated, truncated, info


    def render(self):
        return self.raw_env.render()

    def close(self):
        return self.raw_env.close()


class DiscreteActionWrapper(gym.Env):
    def __init__(self, raw_env):
        super().__init__()
        self.raw_env = raw_env
        self.original_action_space = raw_env.action_space  # Should be MultiDiscrete
        self.obs_space = self.raw_env.observation_space

        # Flatten MultiDiscrete([a, b, c]) → Discrete(a * b * c)
        self.action_space = spaces.Discrete(np.prod(self.original_action_space.nvec))
        self.observation_space = spaces.Box(low=0, high=255, shape=encode_state_cnn(self.raw_env.reset()[0]).shape, dtype=np.uint8)

    def reset(self, **kwargs):
        obs_dict, _ = self.raw_env.reset(**kwargs)
        return encode_state_cnn(obs_dict).astype(np.uint8), {}

    def step(self, flat_action):
        a0, a1, a2 = np.unravel_index(flat_action, self.original_action_space.nvec)
        action = (int(a0), int(a1), int(a2))
        (obs_dict, _), reward, terminated, truncated, info = self.raw_env.step(action)
        flat_obs = encode_state_cnn(obs_dict)
        return flat_obs.astype(np.uint8), reward, terminated, truncated, info

    def render(self, **kwargs):
        return self.raw_env.render(**kwargs)



In [4]:
# ---- Custom CNN Features Extractor ----
class SmallCNN(BaseFeaturesExtractor):
    """
    Custom CNN for small grid inputs.
    Expects observation_space.shape = (C, H, W).
    """
    def __init__(self, observation_space: spaces.Box, features_dim: int = 256):
        super().__init__(observation_space, features_dim)
        height, width, n_input_channels = observation_space.shape

        # Convolutional layers with small kernels
        self.cnn = nn.Sequential(
            nn.Conv2d(n_input_channels, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(128, 32, kernel_size=1),
            nn.ReLU(),
            nn.Flatten(),
        )

        # Compute the flattened dimension
        with th.no_grad():
            sample_input = th.zeros((1, n_input_channels, height, width))
            n_flatten = self.cnn(sample_input).shape[1]

        # Final fully-connected layer
        self.linear = nn.Sequential(
            nn.Linear(n_flatten, features_dim),
            nn.ReLU(),
        )


    def forward(self, observations: th.Tensor) -> th.Tensor:
        # Observations start as (B, H, W, C) and need to be permuted to (B, C, H, W)
        observations = observations.permute(0, 3, 1, 2)  # Change to (B, C, H, W)
        cnn_out = self.cnn(observations)
        return self.linear(cnn_out)


In [5]:
def make_env():
    raw_env = BlockPuzzleEnv(width=12, height=12, num_blocks=3, block_function=random_block)
    wrapped_env = DiscreteActionWrapper(raw_env)

    def mask_fn(env):
        return env.raw_env.game.compute_action_mask()

    masked_env = ActionMasker(wrapped_env, mask_fn)
    monitored_env = Monitor(masked_env)
    return monitored_env

n_envs = 16  # or whatever number you want
vec_env = DummyVecEnv([make_env for _ in range(n_envs)])

policy_kwargs = {
    "features_extractor_class": SmallCNN,
    "features_extractor_kwargs": {"features_dim": 256},
    # If your observations are float32 in [0.0,1.0], disable SB3 normalization:
    "normalize_images": False
}


In [6]:
class AveragedMetricsCallback(BaseCallback):
    def __init__(self, metrics_to_track=None, verbose=0):
        super().__init__(verbose)
        self.metrics_to_track = metrics_to_track or ['cleared_lines', 'invalid_moves', 'move_count']
        self.metric_buffers = {metric: deque(maxlen=100) for metric in self.metrics_to_track}

    def _on_step(self):
        infos = self.locals["infos"]
        for info in infos:
            if "episode" in info:  # Only log at the end of episodes
                for metric in self.metrics_to_track:
                    if metric in info:
                        self.metric_buffers[metric].append(info[metric])

        for metric, buffer in self.metric_buffers.items():
            if buffer:  # Avoid empty
                avg_value = np.mean(buffer)
                self.logger.record(f"custom/{metric}_avg", avg_value)

        return True

In [7]:
model = MaskablePPO(
    policy="CnnPolicy",  # or "CnnPolicy" if your input is image-like
    env=vec_env,
    learning_rate=1e-3,
    n_steps=2048,
    batch_size=128,
    n_epochs=10,
    gamma=0.99,
    verbose=1,
    tensorboard_log="./sb3_logs/", 
    policy_kwargs=policy_kwargs
)

Using cuda device


In [ ]:
model.learn(total_timesteps=1000_000, tb_log_name="PPO_CNN_MASKED", callback=AveragedMetricsCallback())


In [9]:
model.save("sb3_block_ppo_cnn_masked")
